# Notebook 01 — Data Loading & Cleaning
**Dataset:** Brazilian E-Commerce Public Dataset by Olist (Kaggle)  
**Tujuan:** Merge 7 tabel CSV menjadi satu master dataframe yang bersih, lalu simpan ke Parquet.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path('../data/raw')
OUT = Path('../output')
OUT.mkdir(exist_ok=True)

## 1. Load CSV

In [2]:
orders    = pd.read_csv(RAW / 'olist_orders_dataset.csv', parse_dates=[
                'order_purchase_timestamp',
                'order_delivered_customer_date',
                'order_estimated_delivery_date'])
items     = pd.read_csv(RAW / 'olist_order_items_dataset.csv')
payments  = pd.read_csv(RAW / 'olist_order_payments_dataset.csv')
reviews   = pd.read_csv(RAW / 'olist_order_reviews_dataset.csv')
customers = pd.read_csv(RAW / 'olist_customers_dataset.csv')
products  = pd.read_csv(RAW / 'olist_products_dataset.csv')
cat_map   = pd.read_csv(RAW / 'product_category_name_translation.csv')

print('Orders    :', orders.shape)
print('Items     :', items.shape)
print('Payments  :', payments.shape)
print('Reviews   :', reviews.shape)
print('Customers :', customers.shape)
print('Products  :', products.shape)

Orders    : (99441, 8)
Items     : (112650, 7)
Payments  : (103886, 5)
Reviews   : (99224, 7)
Customers : (99441, 5)
Products  : (32951, 9)


## 2. Filter — hanya order berstatus `delivered`

In [3]:
orders_delivered = orders[orders['order_status'] == 'delivered'].copy()
print(f'Delivered orders: {len(orders_delivered):,} dari {len(orders):,} total')

Delivered orders: 96,478 dari 99,441 total


## 3. Agregasi tabel pendukung sebelum merge
Payments dan reviews perlu diagregasi per `order_id` agar tidak duplikat baris saat join.

In [4]:
# Payment: ambil payment_type dominan dan total nilai per order
pay_agg = (
    payments
    .sort_values('payment_value', ascending=False)
    .groupby('order_id')
    .agg(
        payment_type=('payment_type', 'first'),
        payment_value=('payment_value', 'sum')
    )
    .reset_index()
)

# Reviews: ambil satu review score per order (ada duplikat di dataset)
rev_agg = (
    reviews
    .sort_values('review_creation_date', ascending=False)
    .groupby('order_id')
    .agg(review_score=('review_score', 'first'))
    .reset_index()
)

print('pay_agg:', pay_agg.shape)
print('rev_agg:', rev_agg.shape)

pay_agg: (99440, 3)
rev_agg: (98673, 2)


## 4. Merge bertahap

In [5]:
# Tambahkan terjemahan kategori ke products
products_en = products.merge(cat_map, on='product_category_name', how='left')

df = (
    orders_delivered
    .merge(items,       on='order_id',    how='inner')
    .merge(pay_agg,     on='order_id',    how='left')
    .merge(rev_agg,     on='order_id',    how='left')
    .merge(customers,   on='customer_id', how='left')
    .merge(products_en, on='product_id',  how='left')
)

print('Master shape:', df.shape)
df.head(3)

Master shape: (110197, 30)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,customer_state,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1,87285b34884572647811a353c7ac498a,...,SP,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1,595fac2a385ac33a80bd5114aec74eb8,...,BA,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1,aa4383b373c6aca5d8797843e5594415,...,GO,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0,auto


## 5. Feature Engineering

In [6]:
# Revenue per item
df['revenue'] = df['price'] + df['freight_value']

# Bulan pembelian
df['order_month'] = df['order_purchase_timestamp'].dt.to_period('M')
df['order_year']  = df['order_purchase_timestamp'].dt.year
df['order_quarter'] = df['order_purchase_timestamp'].dt.to_period('Q')

# Waktu pengiriman (hari)
df['delivery_days'] = (
    df['order_delivered_customer_date'] - df['order_purchase_timestamp']
).dt.days

# Flag terlambat
df['is_late'] = (
    df['order_delivered_customer_date'] > df['order_estimated_delivery_date']
).astype(int)

# Nama kategori Inggris (fallback ke nama asli jika tidak ada terjemahan)
df['category'] = df['product_category_name_english'].fillna(df['product_category_name'])

print('Kolom baru:', ['revenue','order_month','order_year','order_quarter','delivery_days','is_late','category'])

Kolom baru: ['revenue', 'order_month', 'order_year', 'order_quarter', 'delivery_days', 'is_late', 'category']


## 6. Cek Kualitas Data

In [7]:
cols_check = ['order_id','customer_id','product_id','revenue','order_month',
              'delivery_days','review_score','customer_state','category']

missing = df[cols_check].isnull().sum()
print('Missing values per kolom kunci:')
print(missing[missing > 0] if missing.any() else 'Tidak ada missing value pada kolom kunci.')

Missing values per kolom kunci:
delivery_days       8
review_score      827
category         1537
dtype: int64


## 7. Pilih kolom final & simpan ke Parquet

In [8]:
cols_final = [
    'order_id', 'customer_id', 'product_id', 'seller_id',
    'order_purchase_timestamp', 'order_month', 'order_year', 'order_quarter',
    'price', 'freight_value', 'revenue',
    'payment_type', 'payment_value',
    'review_score',
    'customer_state', 'customer_city',
    'category',
    'delivery_days', 'is_late',
    'order_delivered_customer_date', 'order_estimated_delivery_date'
]

df_master = df[cols_final].copy()
df_master.to_parquet(OUT / 'df_master.parquet', index=False)

print(f'df_master.parquet tersimpan — {len(df_master):,} baris, {df_master.shape[1]} kolom')
df_master.dtypes

df_master.parquet tersimpan — 110,197 baris, 21 kolom


order_id                                    str
customer_id                                 str
product_id                                  str
seller_id                                   str
order_purchase_timestamp         datetime64[us]
order_month                           period[M]
order_year                                int32
order_quarter                     period[Q-DEC]
price                                   float64
freight_value                           float64
revenue                                 float64
payment_type                                str
payment_value                           float64
review_score                            float64
customer_state                              str
customer_city                               str
category                                    str
delivery_days                           float64
is_late                                   int64
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetim